# Recession Forecasting Pipeline — Methodology, PCA, and Results

---

## 1) What your pipeline is doing (and why it’s methodologically solid)

### Goal
- You’re forecasting **NBER recession at $t+12$ months**  
  (target = `USREC` shifted by $-12$).
- That’s a **true forecasting horizon** (harder than contemporaneous classification).
- It’s exactly why your COVID notes are correct: COVID is a sudden break where “slow” macro and yield curve signals can lag.

---

### Data design (good choices)

**Yields**
- Multiple maturities (3m…30y), resampled monthly.

**Recession indicator**
- `USREC` monthly → binary.

**Macro**
- `UNRATE`
- CPI YoY
- `INDPRO` YoY
- `UMCSENT`

**Extras**
- BAA credit spread vs 10Y
- `FEDFUNDS`, etc.

**Labour**
- `PAYEMS` YoY (monthly, high-quality labor-state proxy).

**Term premium**
- `THREEFYTP10` (10y term premium estimate).

---

### Key engineered narrative variables  
*(your “expectations vs term premium” story)*

You’re explicitly testing whether inversion is:
- **expectations-driven** (market pricing future short-rate cuts), vs
- **term-premium-driven** (QE/safe-asset demand compressing the term premium).

You build:
- $y10\_exp\_proxy = y_{10y} - tp_{10y}^{kw}$
- $slope\_{ex\_tp} = (y_{10y} - tp_{10y}^{kw}) - y_{2y}$

**Interpretation**
- That’s conceptually clean: “strip out” term premium from the 10Y and re-check slope behavior.

---

### Anti-leakage / threshold discipline (**big win**)

Your approach is correct:
- Threshold is tuned using **OOF probabilities on TRAIN only**  
  (via expanding `TimeSeriesSplit`).
- Then the final model is trained on the full training window and evaluated on holdout.

**Result**
- That avoids the classic *“I optimized threshold on test”* contamination.

---

## 2) PCA results — what your charts + tables are telling you

You added exactly the missing diagnostics:
- Explained variance ratio
- Loadings table / heatmap
- PC time series
- Rolling-loading drift  

Together they tell a very coherent story.

---

### 2.1 Explained variance (your scree figure)

From `pca_explained_variance_ratio.csv`:
- PC1 ≈ 0.974  
- PC2 ≈ 0.025  
- PC3 ≈ 0.001  

**Interpretation**
- The U.S. yield curve is overwhelmingly **one-dimensional** most of the time:
  - A common **“level” factor** dominates (rates rise/fall together).
- Slope/curvature exist, but they’re small-variance relative to level.
- **Important nuance:**  
  small variance $\neq$ small predictive power.
- Slope can have low variance but still be highly informative about recessions.

---

### 2.2 Loadings table / heatmap  
*(your “maturity × PC” plot)*

Your `pca_loadings_table.csv` shows:

**PC1 (Level)**
- All maturities load positively and similarly (~0.33).
- ✅ That’s textbook **level**.

**PC2 (Slope)**
- Short maturities load negative  
  (3m −0.44, 6m −0.42, 1y −0.29…)
- Long maturities load positive  
  (10y +0.37, 30y +0.53)
- ✅ That’s textbook **slope** (steepening vs flattening).

**PC3 (Curvature)**
- “Ends” more positive-ish
- “Belly” more negative-ish (with sign conventions drifting)
- ✅ Curvature factor: belly moves differently than short/long.

---

### Why this matters for your feature sets

Your `yield_pca` feature set is basically:
- **PC1:** overall rate regime  
  (Volcker era → ZLB/QE → post-COVID hiking cycle)
- **PC2:** curve slope/inversion mechanics  
  (the classic recession channel)
- **PC3:** belly distortions / policy expectations / term premium structure effects

---

### 2.3 PC time series with recession shading  
*(your PC1/PC2/PC3 plot)*

This plot is interpretability gold.

- **PC1 (Level)**
  - Trends down over decades (rates structurally fell)
  - Drops further around ZLB/QE
  - Shifts again post-2020

- **PC2 (Slope)**
  - Lines up with recession risk regimes
  - Swings strongly around tightening → inversion → recession

- **PC3 (Curvature)**
  - Spikes/dips around unusual curve-shape episodes  
    (policy interventions, crisis liquidity, QE, etc.)

**Report framing**
> “Here’s the factor structure, and here’s how factors behave around NBER recessions.”

---

## 3) Rolling PCA loading drift  
*(your three “drift” charts)*

These plots answer:
- **Are the PCA factors stable through time?**

**Answer**
- Mostly stable, but with meaningful regime drift—especially around QE/COVID.

---

### 3.1 PC1 drift (Level)

Your rolling CSV confirms:
- The biggest instability is at the long end.
- `y_30y` loading has huge drift; its minimum occurs around 2020-03 (COVID shock).
- Same story for `y_10y` with a big deformation around COVID.

**Interpretation**
- During crisis + QE-style regimes:
  - The long end can become “special”
  - Term premium compression
  - Flight-to-safety
  - Balance sheet effects

**Implication**
- The “level factor” stops being a perfectly equal-weight movement across maturities.

---

### Practical implication for your model

A single full-sample PCA is fine for baseline, but your rolling drift plots justify:
- Adding term premium controls (you did)
- Doing regime-specific PCA (you also do for some analyses)
- Using rolling PCA scores for real-time forecasting robustness

---

### 3.2 PC2 drift (Slope)

- PC2 stays conceptually “short negative / long positive”
- Weights shift through ZLB/QE and COVID

**Interpretation**
- In ZLB/QE regimes, slope dynamics can reflect:
  - Policy constraints
  - Term premium distortions  
  more than “pure expectations.”

---

### 3.3 PC3 drift (Curvature)

- PC3 shows the most “structural weirdness” (not surprising)
- Curvature is where idiosyncratic maturity segments show up:
  - Belly behavior
  - QE purchase patterns
  - Crisis liquidity

**Conclusion**
- This is exactly why curvature is often fragile across regimes.

---

## 4) Model design: feature sets + algorithms  
*(and what you should say in the report)*

### Feature sets (you built a clean ladder)

- `spreads_only` → interpretable baseline  
- `raw_yields` → nonlinear models can exploit shape directly  
- `yield_pca` → dimensionality reduction + interpretability  
- `macro_only` → “slow economy” state variables  
- `pca_plus_macro` → classic hybrid  
- `pca_macro_extras` → add credit/policy/term premium + your narrative variables  

**Narrative clarity**
> “Does adding macro help? Does adding term premium help? Does PCA help?”

---

### Models

You include:
- Logistic (regularized, `class_weight = balanced`)
- RandomForest
- GradientBoosting
- XGBoost (if available)

**Why this is strong**
- Logistic = interpretable benchmark
- Trees / boosting = nonlinear interactions
- XGB = usually best performer in tabular macro-finance settings

---

## 5) Holdout sensitivity results  
*(from `model_results_holdout.csv`)*

You tested three holdout fractions:
- 0.20
- 0.30
- 0.40  

This is very good practice because recession datasets are small and regime-dependent.

---

### 5.1 `test_frac = 0.20`  
*(most training, smaller test)*

**Top Balanced Accuracy**
- Logistic | `yield_pca` ≈ 0.822 (Thr ≈ 0.844)
- XGBoost | `pca_plus_macro` ≈ 0.803
- XGBoost | `spreads_only` ≈ 0.733

**Interpretation**
- With more training history, PCA factors + linear model generalize well.
- XGBoost is competitive, but threshold behavior can be jumpy on small rare-event samples.

---

### 5.2 `test_frac = 0.30`

**Top**
- XGBoost | `pca_plus_macro` ≈ 0.886 (Thr ≈ 0.046)
- GradBoost | `raw_yields` ≈ 0.846  
  (ROC_AUC ≈ 0.987, PR_AUC ≈ 0.40)

**Interpretation**
- Nonlinear models on raw yields shine here.
- The XGBoost threshold (0.046) screams:
  > “My predicted probabilities are generally low; I need a low cutoff to catch positives.”

---

### 5.3 `test_frac = 0.40`  
*(hardest: least training, biggest test)*

**Top**
- Logistic | `macro_only` ≈ 0.856  
  (ROC_AUC ≈ 0.857, PR_AUC ≈ 0.124)
- Logistic | `pca_macro_extras` ≈ 0.831
- GradBoost | `pca_plus_macro` ≈ 0.818  
  (PR_AUC ≈ 0.347)

**Interpretation**
- Macro variables become more stable than yield-curve shape learning.
- Macro-only Logistic:
  - Recall = 1
  - Precision low  
  → catches recessions but pays with false positives.

---

### Report-ready takeaway

> “Model ranking is sensitive to the holdout window. Raw yields + boosting can excel with moderate training sizes; macro-only becomes relatively robust when training history is reduced. PCA-based models provide consistent baseline performance and strong interpretability.”

---

## 6) Scenario tests: COVID vs GFC  
*(from `model_results_scenarios.csv` + your bar charts)*

This is the most interesting part of your analysis.

---

### 6.1 COVID_test_2019_2021  
*(your F2 bar chart)*

**Top F2**
- XGBoost | `macro_only`  
  F2 ≈ 0.476 (Recall = 1, Precision ≈ 0.154)
- Logistic | `raw_yields`
- XGB | `pca_plus_macro` (~0.45)

**Interpretation**
- COVID “fast shock” dynamics are not well captured by classic slope-only signals.
- Macro-only works because it moves directly in the crisis window.

---

### 6.2 GFC_test_2007_2009  
*(your F2 bar chart)*

**Top F2**
- Logistic | `yield_pca` ≈ 0.60
- RandomForest | `yield_pca` ≈ 0.49
- then XGB | `spreads_only`, etc.

**Interpretation**
- GFC is exactly the type of recession where yield curve structure leads the downturn.
- PCA slope / curvature factors are genuinely informative here.

---

### 6.3 Why PR_AUC = 1.0 does **not** mean perfect classification

You observe cases where:
- PR_AUC ≈ 1.0
- Precision = 0
- Recall = 0

**Why this is not a contradiction**
- PR_AUC is a **ranking metric over all thresholds**.
- Precision/Recall are computed at **your chosen threshold**, not the PR-optimal one.

**Meaning**
- The model ranks positives extremely well,
- But the tuned threshold is too conservative for that scenario.

**Great report point**
> “Ranking metrics (ROC/PR AUC) can look excellent even when classification at a fixed threshold fails. Threshold choice is a separate decision layer.”

---

## 7) Main empirical story

- Yield curve PCA is textbook:
  - PC1 = Level (~97%)
  - PC2 = Slope (~2.5%)
  - PC3 = Curvature (~0.1%)
- PC2 (slope) matters for recession prediction, even if it’s low variance.
- Factor stability is not constant:
  - Rolling loadings drift materially around QE/COVID,
  - Especially at the long end.
- Model performance depends on evaluation regime:
  - **GFC:** PCA-based yield factors perform strongly.
  - **COVID:** macro-only and hybrid models do better.
- Threshold tuning must be discussed explicitly:
  - OOF-train thresholding is correct,
  - Scenario-specific thresholds may differ  
    (PR_AUC vs F2 vs balanced accuracy).


'557ac7e2aa00f4863bf3b8c38457d648'

In [ ]:
# ============================================================
# EC48E — Recession Prediction with Yield Curve + PCA + Macro
# FULL SCRIPT: PCA description + rolling PCA drift + models + valid validation
# (Holdout sensitivity + scenario tests: 2008 vs 2020)
#
# REQUIREMENTS:
#   pip install pandas numpy matplotlib scikit-learn fredapi
# OPTIONAL:
#   pip install xgboost
#
# RUN:
#   export FRED_API_KEY="YOUR_KEY"
#   python ec48e_full.py
# ============================================================

import os
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, average_precision_score
)


warnings.filterwarnings("ignore")

# ----------------------------
# OPTIONAL: XGBoost
# ----------------------------
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

# ----------------------------
# FRED
# ----------------------------
try:
    from fredapi import Fred
    FRED_AVAILABLE = True
except Exception:
    FRED_AVAILABLE = False


# ============================================================
# CONFIG
# ============================================================
@dataclass
class Config:
    start_date: str = "1982-01-01"
    end_date: Optional[str] = None

    horizon_months: int = 12

    # Holdout sensitivity
    test_size_fracs: Tuple[float, ...] = (0.20, 0.30, 0.40)

    # Expanding-window CV inside TRAIN (threshold tuning)
    tscv_splits: int = 6

    # Rolling PCA drift (10 years)
    rolling_pca_window: int = 120     # months
    rolling_pca_step: int = 1         # months

    # Threshold tuning (TRAIN only via OOF)
    threshold_objective: str = "balanced_acc"   # or "f2"
    threshold_grid_n: int = 201
    threshold_min: float = 0.001
    threshold_max: float = 0.999

    # Output
    outdir: str = "ec48e_outputs_full"
    random_state: int = 42

CFG = Config()
os.makedirs(CFG.outdir, exist_ok=True)


COVID_NOTES = """
COVID PREDICTION IMPROVEMENT NOTES (use in report / code comments):
1) COVID was a sudden exogenous shock. Traditional yield curve + slow macro (CPI, IP YoY)
   are low-frequency and react with delay. 12-month-ahead target makes this especially hard.
2) Better features for COVID:
   - High-frequency labour shock: initial claims (weekly), continuing claims
   - Financial stress indices (STLFSI, NFCI), VIX
   - Mobility / real-time activity indicators (Google mobility, OpenTable, TSA, etc.)
   - Credit/liquidity stress: OAS spreads, CP/T-bill spreads, dealer balance sheet indicators
3) Mixed-frequency nowcasting:
   - Aggregate weekly/daily into monthly using "last available" rather than mean
   - MIDAS / mixed-frequency models for proper timing
4) Horizon choice:
   - 12-month ahead for COVID is tricky because shock is immediate.
   - 3–6 months ahead is more realistic for abrupt breaks.
5) Regime/event dummies can help but are not true forecasting unless known ex ante.
   Use them as robustness checks / interpretability aids, not as the main result.
""".strip()


# ============================================================
# FRED HELPERS
# ============================================================
def fred_client() -> "Fred":
    if not FRED_AVAILABLE:
        raise RuntimeError("fredapi not installed. Install with: pip install fredapi")

    api_key = None

    # --- Detect Google Colab ---
    try:
        import google.colab  # noqa: F401
        from google.colab import userdata
        api_key = userdata.get("FREDAPI")
    except Exception:
        pass

    # --- Fallback to environment variable (local / global env) ---
    if not api_key:
        api_key = os.getenv("FREDAPI")

    if not api_key:
        raise RuntimeError(
            "FRED API key not found.\n"
            "Set it via:\n"
            "  • Colab: userdata.set('FREDAPI', 'YOUR_KEY')\n"
            "  • Local: export FREDAPI='YOUR_KEY'"
        )

    return Fred(api_key=api_key)


def fetch_series_monthly_mean(fred: "Fred", series_id: str, start: str, end: Optional[str]) -> pd.Series:
    s = fred.get_series(series_id, observation_start=start, observation_end=end)
    s = pd.Series(s)
    s.index = pd.to_datetime(s.index)
    sm = s.resample("M").mean()
    sm.name = series_id
    return sm

def month_begin_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = (out.index + pd.offsets.MonthBegin(0)).normalize()
    return out


# ============================================================
# REGIME DUMMIES
# ============================================================
def add_regime_dummies(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    idx = out.index

    def in_range(start, end):
        s = pd.to_datetime(start)
        e = pd.to_datetime(end)
        return ((idx >= s) & (idx <= e)).astype(int)

    out["D_GFC"] = in_range("2007-12-01", "2009-06-01")
    out["D_COVID"] = in_range("2020-03-01", "2020-05-01")
    out["D_ZLB_QE"] = in_range("2008-12-01", "2015-12-01")
    return out


# ============================================================
# DATASET BUILD
# ============================================================
def build_dataset(start: str, end: Optional[str]) -> pd.DataFrame:
    fred = fred_client()

    yield_ids = {
        "DGS3MO": "y_3m",
        "DGS6MO": "y_6m",
        "DGS1":   "y_1y",
        "DGS2":   "y_2y",
        "DGS3":   "y_3y",
        "DGS5":   "y_5y",
        "DGS7":   "y_7y",
        "DGS10":  "y_10y",
        "DGS30":  "y_30y",
    }
    rec_id = "USREC"

    macro_ids = {
        "UNRATE":  "unrate",
        "INDPRO":  "indpro",
        "UMCSENT": "umcsent",
        "CPIAUCSL":"cpi",
    }

    extra_ids = {
        "BAA":      "baa",
        "FEDFUNDS": "fedfunds",
    }

    labour_ids = {"PAYEMS": "payems"}
    term_premium_ids = {"THREEFYTP10": "tp_10y_kw"}

    frames = []

    print("Fetching yields...")
    for sid, name in yield_ids.items():
        s = fetch_series_monthly_mean(fred, sid, start, end)
        s.name = name
        frames.append(s)

    print("Fetching recession indicator...")
    rec = fetch_series_monthly_mean(fred, rec_id, start, end)
    rec_flag = (rec > 0.5).astype(int)
    rec_flag.name = "recession"
    frames.append(rec_flag)

    print("Fetching macro variables...")
    for sid, name in macro_ids.items():
        s = fetch_series_monthly_mean(fred, sid, start, end)
        s.name = name
        frames.append(s)

    print("Fetching extras (credit/policy)...")
    for sid, name in extra_ids.items():
        try:
            s = fetch_series_monthly_mean(fred, sid, start, end)
            s.name = name
            frames.append(s)
        except Exception as e:
            print(f"  Warning: could not fetch {sid}: {e}")

    print("Fetching labour market indicator (PAYEMS)...")
    for sid, name in labour_ids.items():
        try:
            s = fetch_series_monthly_mean(fred, sid, start, end)
            s.name = name
            frames.append(s)
        except Exception as e:
            print(f"  Warning: could not fetch {sid}: {e}")

    print("Fetching term premium (THREEFYTP10)...")
    for sid, name in term_premium_ids.items():
        try:
            s = fetch_series_monthly_mean(fred, sid, start, end)
            s.name = name
            frames.append(s)
        except Exception as e:
            print(f"  Warning: could not fetch {sid}: {e}")

    df = pd.concat(frames, axis=1).sort_index()
    df = month_begin_index(df)

    # transforms
    if "cpi" in df.columns:
        df["infl_yoy"] = 100 * (df["cpi"] / df["cpi"].shift(12) - 1.0)
    if "indpro" in df.columns:
        df["indpro_yoy"] = 100 * (df["indpro"] / df["indpro"].shift(12) - 1.0)
    if "payems" in df.columns:
        df["payems_yoy"] = 100 * (df["payems"] / df["payems"].shift(12) - 1.0)

    if "baa" in df.columns and "y_10y" in df.columns:
        df["credit_spread_baa_10y"] = df["baa"] - df["y_10y"]

    if "y_10y" in df.columns and "y_3m" in df.columns:
        df["spread_10y_3m"] = df["y_10y"] - df["y_3m"]
    if "y_10y" in df.columns and "y_2y" in df.columns:
        df["spread_10y_2y"] = df["y_10y"] - df["y_2y"]

    if "fedfunds" in df.columns and "y_2y" in df.columns:
        df["policy_minus_2y"] = df["fedfunds"] - df["y_2y"]

    # term premium / expectations proxies
    if "tp_10y_kw" in df.columns and "y_10y" in df.columns:
        df["y10_exp_proxy"] = df["y_10y"] - df["tp_10y_kw"]
    if "y10_exp_proxy" in df.columns and "y_2y" in df.columns:
        df["slope_ex_tp"] = df["y10_exp_proxy"] - df["y_2y"]

    df = add_regime_dummies(df)
    return df


# ============================================================
# TARGET (t+h)
# ============================================================
def add_forward_target(df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    out = df.copy()
    out = out.dropna(subset=["recession"]).copy()
    out["target"] = out["recession"].shift(-horizon)
    out = out.dropna(subset=["target"]).copy()
    out["target"] = out["target"].astype(int)
    out["recession"] = out["recession"].astype(int)
    return out


# ============================================================
# PCA CORE
# ============================================================
def compute_pca(df: pd.DataFrame, yield_cols: List[str], n_components: int = 3):
    X = df[yield_cols].dropna().copy()
    Xs = StandardScaler().fit_transform(X.values)
    pca = PCA(n_components=n_components, random_state=CFG.random_state).fit(Xs)
    pcs = pca.transform(Xs)

    pc_names = ["PC1_Level", "PC2_Slope", "PC3_Curvature"][:n_components]
    pc_df = pd.DataFrame(pcs, index=X.index, columns=pc_names)

    loadings = pd.DataFrame(
        pca.components_.T,
        index=yield_cols,
        columns=[f"PC{i}" for i in range(1, n_components+1)]
    )
    evr = pd.Series(
        pca.explained_variance_ratio_,
        index=[f"PC{i}" for i in range(1, n_components+1)],
        name="EVR"
    )
    return pc_df, loadings, evr


# ============================================================
# PCA PLOTS
# ============================================================
def shade_recessions(ax, rec_series, alpha=0.12):
    rec = rec_series.fillna(0).astype(int)
    idx = rec.index
    in_rec = False
    start = None
    for t in idx:
        if rec.loc[t] == 1 and not in_rec:
            in_rec, start = True, t
        if rec.loc[t] == 0 and in_rec:
            ax.axvspan(start, t, alpha=alpha)
            in_rec = False
    if in_rec and start is not None:
        ax.axvspan(start, idx[-1], alpha=alpha)

def plot_pc_timeseries(pc_df: pd.DataFrame, rec_series: pd.Series, outpath: str):
    fig, axes = plt.subplots(pc_df.shape[1], 1, figsize=(12, 8), sharex=True)
    if pc_df.shape[1] == 1:
        axes = [axes]
    rec_series = rec_series.reindex(pc_df.index)

    for i, col in enumerate(pc_df.columns):
        ax = axes[i]
        ax.plot(pc_df.index, pc_df[col])
        shade_recessions(ax, rec_series, alpha=0.12)

        # highlight GFC and COVID windows
        ax.axvspan(pd.Timestamp("2007-12-01"), pd.Timestamp("2009-06-01"), alpha=0.15)
        ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-05-01"), alpha=0.15)

        ax.set_title(f"{col} (NBER shading + GFC/COVID highlights)")
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Date")
    plt.tight_layout()
    plt.savefig(outpath, dpi=180)
    plt.close()

def plot_loadings_heatmap(loadings: pd.DataFrame, outpath: str):
    plt.figure(figsize=(8, 4))
    plt.imshow(loadings.values, aspect="auto")
    plt.colorbar()
    plt.xticks(range(loadings.shape[1]), loadings.columns)
    plt.yticks(range(loadings.shape[0]), loadings.index)
    plt.title("PCA Loadings Heatmap (Maturity × PC)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=180)
    plt.close()

def plot_scree(evr: pd.Series, outpath: str):
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(evr)+1), evr.values, marker="o")
    plt.xticks(range(1, len(evr)+1), evr.index)
    plt.title("Explained Variance Ratio (Scree)")
    plt.xlabel("Component")
    plt.ylabel("Explained variance ratio")
    plt.tight_layout()
    plt.savefig(outpath, dpi=180)
    plt.close()


# ============================================================
# ROLLING PCA DRIFT
# ============================================================
def rolling_pca_loadings(df: pd.DataFrame, yield_cols: List[str], window_months: int = 120, step: int = 1):
    Xfull = df[yield_cols].dropna()
    pc_loadings = {1: [], 2: [], 3: []}
    times = []

    for end_i in range(window_months, len(Xfull)+1, step):
        w = Xfull.iloc[end_i-window_months:end_i]
        t = w.index[-1]

        Xs = StandardScaler().fit_transform(w.values)
        pca = PCA(n_components=3, random_state=CFG.random_state).fit(Xs)
        L = pca.components_.T

        # sign conventions (stable drift plots)
        if L[:,0].mean() < 0: L[:,0] *= -1
        if L[-1,1] < 0:       L[:,1] *= -1
        if L[-1,2] < 0:       L[:,2] *= -1

        for k in [1,2,3]:
            pc_loadings[k].append(pd.Series(L[:,k-1], index=yield_cols))
        times.append(t)

    out = {}
    for k in [1,2,3]:
        out[f"PC{k}"] = pd.DataFrame(pc_loadings[k], index=pd.DatetimeIndex(times))
    return out

def plot_rolling_drift(roll_loadings: Dict[str, pd.DataFrame], outprefix: str):
    for pc_name, Ldf in roll_loadings.items():
        plt.figure(figsize=(12, 5))
        for col in Ldf.columns:
            plt.plot(Ldf.index, Ldf[col], label=col)

        # QE/ZLB highlight
        plt.axvspan(pd.Timestamp("2008-12-01"), pd.Timestamp("2015-12-01"), alpha=0.12)
        plt.title(f"Rolling PCA Loading Drift — {pc_name} (window={CFG.rolling_pca_window}m)")
        plt.xlabel("Window end date")
        plt.ylabel("Loading")
        plt.grid(True, alpha=0.3)
        plt.legend(ncol=3, fontsize=7)
        plt.tight_layout()
        plt.savefig(f"{outprefix}_{pc_name}.png", dpi=180)
        plt.close()


# ============================================================
# FEATURE SETS
# ============================================================
def make_feature_sets(df: pd.DataFrame, pc_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    yield_cols  = [c for c in ["y_3m","y_6m","y_1y","y_2y","y_3y","y_5y","y_7y","y_10y","y_30y"] if c in df.columns]
    spread_cols = [c for c in ["spread_10y_3m","spread_10y_2y"] if c in df.columns]

    macro_cols  = [c for c in ["unrate","infl_yoy","indpro_yoy","umcsent","payems_yoy"] if c in df.columns]
    extra_cols  = [c for c in ["fedfunds","credit_spread_baa_10y","tp_10y_kw","policy_minus_2y","y10_exp_proxy","slope_ex_tp"] if c in df.columns]
    regime_cols = [c for c in ["D_GFC","D_COVID","D_ZLB_QE"] if c in df.columns]

    fsets: Dict[str, pd.DataFrame] = {}

    if spread_cols:
        fsets["spreads_only"] = df[spread_cols + ["target"]].copy()

    if len(yield_cols) >= 5:
        fsets["raw_yields"] = df[yield_cols + ["target"]].copy()

    # yield_pca = PC scores + spreads
    if pc_df is not None and pc_df.shape[1] >= 3:
        tmp = pc_df.copy()
        for s in spread_cols:
            tmp[s] = df[s].reindex(tmp.index)
        tmp["target"] = df["target"].reindex(tmp.index)
        fsets["yield_pca"] = tmp

    # macro only (plus regime dummies)
    if macro_cols:
        base = df[macro_cols].copy()
        for r in regime_cols:
            base[r] = df[r]
        base["target"] = df["target"]
        fsets["macro_only"] = base

    # pca + macro (+ regime)
    if "yield_pca" in fsets and macro_cols:
        base = fsets["yield_pca"].drop(columns=["target"]).copy()
        for m in macro_cols:
            base[m] = df[m].reindex(base.index)
        for r in regime_cols:
            base[r] = df[r].reindex(base.index)
        base["target"] = df["target"].reindex(base.index)
        fsets["pca_plus_macro"] = base

    # pca + macro + extras
    if "pca_plus_macro" in fsets and extra_cols:
        base = fsets["pca_plus_macro"].drop(columns=["target"]).copy()
        for e in extra_cols:
            base[e] = df[e].reindex(base.index)
        base["target"] = df["target"].reindex(base.index)
        fsets["pca_macro_extras"] = base

    for k in list(fsets.keys()):
        fsets[k] = fsets[k].dropna().copy()

    return fsets


# ============================================================
# MODELS
# ============================================================
def define_models() -> Dict[str, object]:
    models: Dict[str, object] = {}

    models["Logistic"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=CFG.random_state
        ))
    ])

    models["RandomForest"] = RandomForestClassifier(
        n_estimators=600,
        max_depth=6,
        min_samples_leaf=6,
        class_weight="balanced_subsample",
        random_state=CFG.random_state,
        n_jobs=-1
    )

    models["GradBoost"] = GradientBoostingClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=3,
        random_state=CFG.random_state
    )

    if XGB_AVAILABLE:
        models["XGBoost"] = XGBClassifier(
            n_estimators=600,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=CFG.random_state
        )
    return models


# ============================================================
# METRICS + THRESHOLDS (robust)
# ============================================================
def predict_proba(model, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)[:, 1]
        return np.asarray(p, dtype=float)
    return np.asarray(model.predict(X), dtype=float)

def fbeta_from_counts(tp, fp, fn, beta=2.0) -> float:
    beta2 = beta**2
    denom = (1+beta2)*tp + beta2*fn + fp
    return 0.0 if denom == 0 else (1+beta2)*tp/denom

def compute_metrics(y_true, y_prob, thr=0.5) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    finite = np.isfinite(y_true) & np.isfinite(y_prob)
    y_true = y_true[finite]
    y_prob = y_prob[finite]

    if len(y_true) == 0:
        return {k: np.nan for k in ["Accuracy","BalancedAcc","Precision","Recall","F1","F2","ROC_AUC","PR_AUC"]}

    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAcc": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "F2": fbeta_from_counts(tp, fp, fn, beta=2.0),
    }

    if len(np.unique(y_true)) == 2:
        out["ROC_AUC"] = roc_auc_score(y_true, y_prob)
        out["PR_AUC"] = average_precision_score(y_true, y_prob)
    else:
        out["ROC_AUC"] = np.nan
        out["PR_AUC"] = np.nan

    return out

def optimize_threshold(y_true, y_prob, objective="balanced_acc") -> Tuple[float, float]:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    finite = np.isfinite(y_true) & np.isfinite(y_prob)
    y_true = y_true[finite]
    y_prob = y_prob[finite]

    grid = np.linspace(CFG.threshold_min, CFG.threshold_max, CFG.threshold_grid_n)
    best_thr, best_val = 0.5, -np.inf

    for thr in grid:
        y_pred = (y_prob >= thr).astype(int)

        if objective == "balanced_acc":
            val = balanced_accuracy_score(y_true, y_pred)
        elif objective == "f2":
            tp = int(((y_true == 1) & (y_pred == 1)).sum())
            fp = int(((y_true == 0) & (y_pred == 1)).sum())
            fn = int(((y_true == 1) & (y_pred == 0)).sum())
            val = fbeta_from_counts(tp, fp, fn, beta=2.0)
        else:
            raise ValueError("objective must be 'balanced_acc' or 'f2'")

        if val > best_val:
            best_val, best_thr = float(val), float(thr)

    return best_thr, best_val


def oof_probs_train_only(X_train: pd.DataFrame, y_train: pd.Series, model_name: str, model_obj) -> np.ndarray:
    """OOF probabilities inside TRAIN for threshold tuning (no leakage)."""
    mask = X_train.notna().all(axis=1) & y_train.notna()
    X_train = X_train.loc[mask].copy()
    y_train = y_train.loc[mask].copy()

    tscv = TimeSeriesSplit(n_splits=CFG.tscv_splits)
    oof = np.full(len(X_train), np.nan, dtype=float)

    for tr, te in tscv.split(X_train):
        Xtr, Xte = X_train.iloc[tr], X_train.iloc[te]
        ytr = y_train.iloc[tr].values

        if len(np.unique(ytr)) < 2:
            continue

        mdl = model_obj
        if model_name == "XGBoost":
            pos = ytr.sum()
            neg = len(ytr) - pos
            spw = (neg / max(pos, 1))
            mdl = XGBClassifier(**{**model_obj.get_params(), "scale_pos_weight": spw})

        try:
            mdl.fit(Xtr, ytr)
            p = predict_proba(mdl, Xte)
            if np.any(~np.isfinite(p)):
                continue
            oof[te] = p
        except Exception:
            continue

    base = float(y_train.mean()) if len(y_train) else 0.0
    oof = np.where(np.isnan(oof), base, oof)
    return oof


# ============================================================
# VALIDATION SPLITS
# ============================================================
def holdout_split(df: pd.DataFrame, test_size_frac: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    n = len(df)
    cut = int(np.floor((1.0 - test_size_frac) * n))
    return df.iloc[:cut].copy(), df.iloc[cut:].copy()

def scenario_split(df: pd.DataFrame, train_end: str, test_start: str, test_end: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = df.loc[:pd.to_datetime(train_end)].copy()
    test  = df.loc[pd.to_datetime(test_start):pd.to_datetime(test_end)].copy()
    return train, test


# ============================================================
# MAIN
# ============================================================
def main():
    print("============================================================")
    print("EC48E Recession Prediction — FULL PIPELINE")
    print("============================================================")
    print(f"Threshold objective = {CFG.threshold_objective} | grid_n={CFG.threshold_grid_n}\n")
    print(COVID_NOTES + "\n")

    df_raw = build_dataset(CFG.start_date, CFG.end_date)
    df = add_forward_target(df_raw, CFG.horizon_months)

    print(f"Data window: {df.index.min().date()} -> {df.index.max().date()}")
    print(f"Observations (months): {len(df)}")
    print(f"Recession frequency (current month): {df['recession'].mean():.2%}")
    print(f"Target frequency (t+{CFG.horizon_months}): {df['target'].mean():.2%}\n")

    # --- PCA DESCRIPTION BLOCK (Step A)
    yield_cols = [c for c in ["y_3m","y_6m","y_1y","y_2y","y_3y","y_5y","y_7y","y_10y","y_30y"] if c in df.columns]
    if len(yield_cols) < 5:
        raise RuntimeError("Not enough yield columns fetched for PCA.")

    pc_df, loadings, evr = compute_pca(df, yield_cols, n_components=3)

    # Save PCA tables
    loadings.to_csv(os.path.join(CFG.outdir, "pca_loadings_table.csv"))
    evr.to_csv(os.path.join(CFG.outdir, "pca_explained_variance_ratio.csv"))

    # PCA plots
    plot_pc_timeseries(pc_df, df["recession"], os.path.join(CFG.outdir, "fig_PC_timeseries.png"))
    plot_loadings_heatmap(loadings, os.path.join(CFG.outdir, "fig_PCA_loadings_heatmap.png"))
    plot_scree(evr, os.path.join(CFG.outdir, "fig_PCA_scree.png"))

    # Rolling PCA drift (optional but powerful)
    roll = rolling_pca_loadings(df, yield_cols, window_months=CFG.rolling_pca_window, step=CFG.rolling_pca_step)
    for k, v in roll.items():
        v.to_csv(os.path.join(CFG.outdir, f"rolling_loadings_{k}.csv"))
    plot_rolling_drift(roll, os.path.join(CFG.outdir, "fig_rolling_loading_drift"))

    # --- Feature sets (Step B/C)
    feature_sets = make_feature_sets(df, pc_df)
    print(f"Feature sets created: {list(feature_sets.keys())}\n")

    models = define_models()
    print(f"Models: {list(models.keys())}\n")

    # =========================================================
    # HOLDOUT SENSITIVITY RUNS
    # =========================================================
    holdout_rows = []

    for test_frac in CFG.test_size_fracs:
        print("============================================================")
        print(f"HOLDOUT SENSITIVITY RUN: test_size_frac={test_frac:.2f}")
        print("============================================================")

        for fset, dff in feature_sets.items():
            train_df, test_df = holdout_split(dff, test_size_frac=test_frac)

            Xtr = train_df.drop(columns=["target"]).copy()
            ytr = train_df["target"].astype(int).copy()
            Xte = test_df.drop(columns=["target"]).copy()
            yte = test_df["target"].astype(int).values

            if len(train_df) < 120 or len(test_df) < 12:
                continue
            if ytr.nunique() < 2:
                continue

            print("\n------------------------------------------------------------")
            print(f"Feature set: {fset} | train={len(train_df)} test={len(test_df)} | pos(train)={ytr.mean():.2%}")

            for mname, mobj in models.items():
                # threshold on TRAIN only using OOF
                oof_prob = oof_probs_train_only(Xtr, ytr, mname, mobj)
                thr_opt, thr_val = optimize_threshold(ytr.values, oof_prob, objective=CFG.threshold_objective)

                mdl = mobj
                if mname == "XGBoost":
                    pos = ytr.values.sum()
                    neg = len(ytr) - pos
                    spw = (neg / max(pos, 1))
                    mdl = XGBClassifier(**{**mobj.get_params(), "scale_pos_weight": spw})

                try:
                    mdl.fit(Xtr, ytr.values)
                    prob_te = predict_proba(mdl, Xte)
                    met = compute_metrics(yte, prob_te, thr=thr_opt)
                except Exception:
                    continue

                print(f"  {mname:12s} | thr={thr_opt:.3f} | "
                      f"ROC_AUC={met['ROC_AUC']:.3f} | PR_AUC={met['PR_AUC']:.3f} | "
                      f"BalAcc={met['BalancedAcc']:.3f} | F2={met['F2']:.3f}")

                holdout_rows.append({
                    "TestFrac": test_frac,
                    "FeatureSet": fset,
                    "Model": mname,
                    "ThrObjective": CFG.threshold_objective,
                    "ThrOpt": thr_opt,
                    "ThrObjectiveValue_onTrainOOF": thr_val,
                    **met
                })

    holdout_df = pd.DataFrame(holdout_rows)
    holdout_path = os.path.join(CFG.outdir, "model_results_holdout.csv")
    holdout_df.to_csv(holdout_path, index=False)
    print(f"\nSaved holdout results: {holdout_path}")

    # =========================================================
    # SCENARIO TESTS: GFC vs COVID (your “convince prof” section)
    # =========================================================
    scenarios = {
        "GFC_test_2007_2009": ("2006-12-01", "2007-01-01", "2009-12-01"),
        "COVID_test_2019_2021": ("2018-12-01", "2019-01-01", "2021-12-01"),
    }

    scenario_rows = []

    print("\n============================================================")
    print("SCENARIO TESTS (GFC vs COVID) — time-respecting splits")
    print("============================================================")

    for scen_name, (train_end, test_start, test_end) in scenarios.items():
        print(f"\n--- Scenario: {scen_name} | train<= {train_end} | test: {test_start}..{test_end}")

        for fset, dff in feature_sets.items():
            train_df, test_df = scenario_split(dff, train_end, test_start, test_end)
            if len(train_df) < 120 or len(test_df) < 12:
                continue

            Xtr = train_df.drop(columns=["target"]).copy()
            ytr = train_df["target"].astype(int).copy()
            Xte = test_df.drop(columns=["target"]).copy()
            yte = test_df["target"].astype(int).values

            if ytr.nunique() < 2:
                continue

            for mname, mobj in models.items():
                oof_prob = oof_probs_train_only(Xtr, ytr, mname, mobj)
                thr_opt, thr_val = optimize_threshold(ytr.values, oof_prob, objective=CFG.threshold_objective)

                mdl = mobj
                if mname == "XGBoost":
                    pos = ytr.values.sum()
                    neg = len(ytr) - pos
                    spw = (neg / max(pos, 1))
                    mdl = XGBClassifier(**{**mobj.get_params(), "scale_pos_weight": spw})

                try:
                    mdl.fit(Xtr, ytr.values)
                    prob_te = predict_proba(mdl, Xte)
                    met = compute_metrics(yte, prob_te, thr=thr_opt)
                except Exception:
                    continue

                scenario_rows.append({
                    "Scenario": scen_name,
                    "TrainEnd": train_end,
                    "TestStart": test_start,
                    "TestEnd": test_end,
                    "FeatureSet": fset,
                    "Model": mname,
                    "ThrObjective": CFG.threshold_objective,
                    "ThrOpt": thr_opt,
                    "ThrObjectiveValue_onTrainOOF": thr_val,
                    **met
                })

    scen_df = pd.DataFrame(scenario_rows)
    scen_path = os.path.join(CFG.outdir, "model_results_scenarios.csv")
    scen_df.to_csv(scen_path, index=False)
    print(f"\nSaved scenario results: {scen_path}")

    # Quick “prof-friendly” plots: PR_AUC and F2 by scenario
    if not scen_df.empty:
        for metric in ["PR_AUC", "F2"]:
            for scen_name in scen_df["Scenario"].unique():
                sub = scen_df[scen_df["Scenario"] == scen_name].copy()
                if sub.empty:
                    continue

                # best per (featureset, model) already; just plot all
                sub = sub.sort_values(metric, ascending=False).head(15)

                plt.figure(figsize=(10, 5))
                labels = sub["Model"] + " | " + sub["FeatureSet"]
                plt.bar(range(len(sub)), sub[metric].values)
                plt.xticks(range(len(sub)), labels, rotation=45, ha="right", fontsize=8)
                plt.title(f"Top results by {metric} — {scen_name}")
                plt.tight_layout()
                plt.savefig(os.path.join(CFG.outdir, f"fig_top_{metric}_{scen_name}.png"), dpi=180)
                plt.close()

    print(f"\nAll outputs in: {CFG.outdir}")


if __name__ == "__main__":
    main()

EC48E Recession Prediction — FULL PIPELINE
Threshold objective = balanced_acc | grid_n=201

COVID PREDICTION IMPROVEMENT NOTES (use in report / code comments):
1) COVID was a sudden exogenous shock. Traditional yield curve + slow macro (CPI, IP YoY)
   are low-frequency and react with delay. 12-month-ahead target makes this especially hard.
2) Better features for COVID:
   - High-frequency labour shock: initial claims (weekly), continuing claims
   - Financial stress indices (STLFSI, NFCI), VIX
   - Mobility / real-time activity indicators (Google mobility, OpenTable, TSA, etc.)
   - Credit/liquidity stress: OAS spreads, CP/T-bill spreads, dealer balance sheet indicators
3) Mixed-frequency nowcasting:
   - Aggregate weekly/daily into monthly using "last available" rather than mean
   - MIDAS / mixed-frequency models for proper timing
4) Horizon choice:
   - 12-month ahead for COVID is tricky because shock is immediate.
   - 3–6 months ahead is more realistic for abrupt breaks.
5) Regim